# The Data Stack
### NumPy, pandas, Matplotlib & OpenCV for ML

18 runnable, **seeded** programs — the four-library spine under every real ML project. Each program is seeded with `np.random.default_rng(7)`, runs offline (no downloads, no Kaggle), and **reproduces to the digit**: the number on screen equals the terminal equals the headline.

Run the setup cell once, then any episode top-to-bottom.

In [ ]:
!pip install -q numpy pandas matplotlib opencv-python

## Setup · write every episode program to disk

In [ ]:
%%writefile ep01_vectorization.py
"""
DS01 · Arrays, dtypes & Vectorization — seeded, offline, reproduces to the digit.
One array op replaces a Python loop and returns the bit-for-bit identical result.
"""
import pathlib
import numpy as np

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds01.png"


def make_data():
    rng = np.random.default_rng(7)                 # the contract: same seed → same numbers
    return rng.normal(loc=50, scale=12, size=20)   # 20 samples, float64


def standardize_loop(x):
    mu, sd = x.mean(), x.std()
    out = [(v - mu) / sd for v in x]               # the for-loop everyone reaches for
    return np.array(out)


def standardize_vec(x):
    return (x - x.mean()) / x.std()                # one array op, no loop


if __name__ == "__main__":
    x = make_data()
    print("dtype:", x.dtype)
    print("shape:", x.shape)
    print("raw   mean:", round(float(x.mean()), 3), " std:", round(float(x.std()), 4))

    z_loop, z_vec = standardize_loop(x), standardize_vec(x)
    assert np.array_equal(z_loop, z_vec), "loop and vectorized disagree!"
    print("array_equal(loop, vec):", np.array_equal(z_loop, z_vec))

    z = z_vec
    z_mean, z_std = round(float(z.mean()), 1), round(float(z.std()), 1)
    print("std   mean:", z_mean, " std:", z_std)

    # reproduce-to-the-digit: run again, must match exactly
    assert np.array_equal(make_data(), x), "rerun: data drifted"
    assert np.array_equal(standardize_vec(make_data()), z), "rerun: result drifted"
    print("reproduce check: PASS")
    assert (z_mean, z_std) == (0.0, 1.0)

    # ---- hero figure: before/after histogram (raw vs standardized) ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"axes.edgecolor": "#7d8693", "text.color": "#1b2330",
                         "axes.labelcolor": "#1b2330", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.4))
    ax1.hist(x, bins=8, color="#2BD9FF", edgecolor="#1b2330")
    ax1.axvline(float(x.mean()), color="#FF5C6C", lw=2)
    ax1.set_title(f"raw  (mean≈{round(float(x.mean()),1)}, std≈{round(float(x.std()),1)})", fontsize=10)
    ax2.hist(z, bins=8, color="#4DE0A0", edgecolor="#1b2330")
    ax2.axvline(0.0, color="#FF5C6C", lw=2)
    ax2.set_title(f"standardized  (mean={z_mean}, std={z_std})", fontsize=10)
    fig.suptitle("one array op == the loop  ·  mean → 0, std → 1", fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130)
    print("saved:", FIG.name)


In [ ]:
%%writefile ep02_broadcasting.py
"""
DS02 · Broadcasting — shapes that align without copies. Seeded, offline, reproduces to the digit.
A column (4,1) times a row (3,) broadcasts to a full (4,3) table in one expression — no loop, no copy.
"""
import pathlib
import numpy as np

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds02.png"


def make_vectors():
    rng = np.random.default_rng(7)        # the course contract: same seed → same numbers
    a = rng.integers(1, 6, size=4)        # (4,)  values 1..5
    b = rng.integers(1, 6, size=3)        # (3,)  values 1..5
    return a, b


def broadcast_table(a, b):
    col = a.reshape(-1, 1)                 # (4,1)  -1 = "infer this length"
    return col * b                        # (4,1)*(3,) -> (4,3), no loop, no copy


def loop_table(a, b):
    out = np.empty((len(a), len(b)), dtype=np.int64)
    for i in range(len(a)):
        for j in range(len(b)):
            out[i, j] = a[i] * b[j]       # the double loop broadcasting replaces
    return out


if __name__ == "__main__":
    a, b = make_vectors()
    col = a.reshape(-1, 1)
    table = broadcast_table(a, b)
    print("a =", a, "shape", a.shape)
    print("b =", b, "shape", b.shape)
    print("col shape", col.shape)
    print("table shape", table.shape)
    print(table)

    loop = loop_table(a, b)
    assert np.array_equal(table, loop), "broadcast must equal the loop"
    print("match loop:", np.array_equal(table, loop))

    a2, b2 = make_vectors()               # reproduce: re-seed, identical draws
    assert np.array_equal(a, a2) and np.array_equal(b, b2)
    print("reproduces:", np.array_equal(a, a2) and np.array_equal(b, b2))

    max_cell = int(table.max())
    i, j = np.unravel_index(int(table.argmax()), table.shape)
    total = int(table.sum())
    print("max cell:", max_cell)
    print("max at (row,col): (%d, %d) = a[%d]*b[%d] = %d*%d" % (i, j, i, j, a[i], b[j]))
    print("total sum:", total)

    # ---- hero figure: white card, 4x3 heatmap of the broadcasted table ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    plt.rcParams.update({"axes.edgecolor": "#7d8693", "text.color": "#1b2330",
                         "axes.labelcolor": "#1b2330", "xtick.color": "#1b2330", "ytick.color": "#1b2330"})
    cyan = LinearSegmentedColormap.from_list("cyanramp", ["#EAFBFF", "#2BD9FF"])
    fig, ax = plt.subplots(figsize=(8, 3.4))
    im = ax.imshow(table, cmap=cyan, aspect="auto")
    for r in range(table.shape[0]):
        for c in range(table.shape[1]):
            ax.text(c, r, str(int(table[r, c])), ha="center", va="center",
                    fontsize=15, fontweight="bold", color="#10202b")
    ax.set_xticks(range(3)); ax.set_xticklabels([f"b={v}" for v in b], fontsize=10)
    ax.set_yticks(range(4)); ax.set_yticklabels([f"a={v}" for v in a], fontsize=10)
    ax.set_xlabel("b  (3,)  — row operand", fontsize=10)
    ax.set_ylabel("col  (4,1)  — column operand", fontsize=10)
    # outline the max cell in green
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="#2BB57A", lw=3))
    ax.set_title(f"(4,1) × (3,) → (4,3)   ·   max {max_cell} = a[{i}]·b[{j}]   ·   sum {total}",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130)
    print("saved:", FIG.name)


In [ ]:
%%writefile ep03_masks.py
"""
DS03 · Fancy Indexing & Boolean Masks — select without loops. Seeded, offline, reproduces to the digit.
A boolean mask is a query over an array: x > 50 returns a True/False grid, x[mask] pulls the passing cells.
"""
import pathlib
import numpy as np

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds03.png"
THR = 50


def make_grid():
    rng = np.random.default_rng(7)             # the course contract: same seed → same numbers
    return rng.integers(0, 100, size=(4, 5))   # 4x5 grid, values 0..99


if __name__ == "__main__":
    x = make_grid()
    mask = x > THR                             # same shape, True where cell passes
    selected = x[mask]                         # flat array of passing values, no loop
    count = int(mask.sum())                    # True==1, False==0 → counts matches
    sel_sum = int(selected.sum())
    clamped = np.where(mask, THR, x)           # >thr → 50, else keep original

    rows = np.array([0, 2, 3])
    cols = np.array([1, 4, 0])
    picked = x[rows, cols]                      # fancy index: pairs (0,1),(2,4),(3,0)

    print("array:"); print(x)
    print("mask:"); print(mask)
    print("count > %d : %d" % (THR, count))
    print("selected  :", selected)
    print("sel_sum   :", sel_sum)
    print("clamped:"); print(clamped)
    print("picked    :", picked, "pick_sum :", int(picked.sum()))

    x2 = make_grid()                           # reproduce-to-the-digit
    assert np.array_equal(x, x2), "not reproducible"
    assert count == 11 and sel_sum == 869 and int(picked.sum()) == 193
    print("reproducible:", True, "| figure lit == count:", count == int(mask.sum()))

    # ---- hero figure: white card, 4x5 grid, mask-lit cells (cyan where x>50) ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#7d8693",
                         "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.set_xlim(0, 5); ax.set_ylim(0, 4); ax.invert_yaxis(); ax.set_aspect("auto")
    for r in range(4):
        for c in range(5):
            on = bool(mask[r, c])
            ax.add_patch(plt.Rectangle((c, r), 1, 1,
                         facecolor="#2BD9FF" if on else "#FFFFFF",
                         alpha=0.30 if on else 1.0,
                         edgecolor="#2BD9FF" if on else "#ececec",
                         lw=2 if on else 1))
            ax.text(c + 0.5, r + 0.5, str(int(x[r, c])), ha="center", va="center",
                    fontsize=15, fontweight="bold" if on else "normal",
                    color="#10202b" if on else "#8A8DA8")
    ax.set_xticks([v + 0.5 for v in range(5)]); ax.set_xticklabels(range(5))
    ax.set_yticks([v + 0.5 for v in range(4)]); ax.set_yticklabels(range(4))
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_color("#cfd4da")
    ax.set_title(f"a mask is a query · {count} of 20 cells pass  x > {THR}  ·  sum {sel_sum}",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130)
    print("saved:", FIG.name)


In [ ]:
%%writefile ep04_linalg.py
"""
DS04 · Linear Algebra for ML — dot, matmul, norms, lstsq. Seeded, offline, reproduces to the digit.
Fitting a line is not a special algorithm: stack a design matrix, call np.linalg.lstsq once, get the weights.
"""
import pathlib
import numpy as np

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds04.png"
N = 40
TRUE_SLOPE, TRUE_INTERCEPT = 2.3, 1.0


def make_data():
    rng = np.random.default_rng(7)               # the course contract: same seed → same numbers
    x = rng.uniform(0, 10, size=N)               # inputs in [0,10)
    noise = rng.normal(0, 1.5, size=N)           # messy, like real data
    y = TRUE_SLOPE * x + TRUE_INTERCEPT + noise
    return x, y


def fit(x, y):
    X = np.column_stack([x, np.ones(len(x))])    # design matrix (40,2)
    w, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
    return X, w, residuals, rank


if __name__ == "__main__":
    x, y = make_data()
    X, w, residuals, rank = fit(x, y)
    slope, intercept = w
    y_hat = X @ w                                # predictions: one matmul
    resid_norm = float(np.linalg.norm(y - y_hat))

    print("X shape:", X.shape)
    print("rank:", int(rank))
    print("slope     = %.4f" % slope)
    print("intercept = %.4f" % intercept)
    print("residual norm = %.4f" % resid_norm)

    assert np.isclose(np.sqrt(residuals[0]), resid_norm), "norm mismatch"
    print("sqrt(lstsq residual) matches norm:", True)

    x2, y2 = make_data()                         # reproduce: identical weights
    _, w2, _, _ = fit(x2, y2)
    print("reproduces:", np.array_equal(w, w2))
    assert np.allclose(X @ w, y_hat)

    fig_slope, fig_intercept = round(float(slope), 4), round(float(intercept), 4)
    print("figure line: y = %sx + %s" % (fig_slope, fig_intercept))

    # ---- hero figure: scatter + lstsq line on white card ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#7d8693",
                         "grid.color": "#ececec"})
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.scatter(x, y, s=34, color="#2BD9FF", edgecolor="#10202b", linewidth=0.5, zorder=3, label="40 seeded points")
    xs = np.array([x.min(), x.max()])
    ax.plot(xs, slope * xs + intercept, color="#5FBF1F", lw=2.6, zorder=4,
            label="lstsq line: y = %.3fx + %.3f" % (slope, intercept))
    ax.grid(True, lw=0.7); ax.set_axisbelow(True)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
    ax.text(0.97, 0.06, "residual norm = %.4f" % resid_norm, transform=ax.transAxes,
            ha="right", va="bottom", fontsize=10, color="#2BB57A", fontweight="bold")
    ax.set_title("least squares = one matrix call  ·  slope %.4f  ·  intercept %.4f" % (slope, intercept),
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130)
    print("saved:", FIG.name)


In [ ]:
%%writefile ep05_dataframe.py
"""
DS05 · DataFrame & Series — the labeled array. Seeded, offline, reproduces to the digit.
A pandas DataFrame is numpy with names + mixed dtypes; head/dtypes/describe read it before you trust it.
"""
import pathlib
import numpy as np
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds05.png"
N = 6


def build():
    rng = np.random.default_rng(7)                  # the course contract: same seed → same numbers
    return pd.DataFrame({
        "user_id": np.arange(1001, 1001 + N),       # int64
        "age":     rng.integers(18, 65, size=N),     # int64
        "score":   rng.normal(70, 12, size=N).round(2),  # float64
        "active":  rng.random(N) > 0.5,              # bool
        "plan":    rng.choice(["free", "pro", "team"], size=N),  # str (text)
    })


if __name__ == "__main__":
    df = build()
    print("shape:", df.shape)
    print("\nhead:"); print(df.head())
    print("\ndtypes:"); print(df.dtypes)
    print("\ndescribe (numeric):"); print(df.describe().round(2))

    df2 = build()                                   # reproduce: rebuild from same seed
    assert df.equals(df2), "not reproducible"

    mean_score = round(float(df["score"].mean()), 2)
    assert mean_score == round(float(df.describe().loc["mean", "score"]), 2) == 67.14
    print("\nreproduce: OK | figure mean(score) == terminal mean(score) ==", mean_score)

    # ---- hero figure: head() table + dtype chips on white card ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3.8)); ax.axis("off")
    cols = ["", "user_id", "age", "score", "active", "plan"]
    cw = [0.09, 0.19, 0.13, 0.18, 0.19, 0.22]       # column width fractions
    cx = np.cumsum([0] + cw)
    head = df.head()
    rows = [[str(i)] + [str(head.iloc[i][c]) for c in df.columns] for i in range(5)]
    rh = 0.135; top = 0.92
    # header
    for k in range(6):
        ax.add_patch(plt.Rectangle((cx[k], top - rh), cw[k], rh, transform=ax.transAxes,
                     facecolor="#FF4D9D" if k > 0 else "#FFFFFF", alpha=0.9 if k > 0 else 1,
                     edgecolor="#cfd4da", lw=0.8))
        ax.text(cx[k] + cw[k] / 2, top - rh / 2, cols[k], transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="#FFFFFF" if k > 0 else "#7d8693", family="monospace")
    # data rows
    for r in range(5):
        for k in range(6):
            y = top - rh * (r + 2)
            hot = (k == 3 and r == 4)                # max score cell 86.08
            ax.add_patch(plt.Rectangle((cx[k], y), cw[k], rh, transform=ax.transAxes,
                         facecolor="#9EFF3D" if hot else "#FFFFFF", alpha=0.5 if hot else 1,
                         edgecolor="#ececec", lw=0.8))
            ax.text(cx[k] + cw[k] / 2, y + rh / 2, rows[r][k], transform=ax.transAxes, ha="center", va="center",
                    fontsize=11, color="#8A8DA8" if k == 0 else "#1b2330",
                    fontweight="bold" if hot else "normal", family="monospace")
    # dtype chips
    chips = [("user_id · int64", "#2BD9FF"), ("age · int64", "#2BD9FF"), ("score · float64", "#9EFF3D"),
             ("active · bool", "#4DE0A0"), ("plan · str", "#FF9F40")]
    n = len(chips); gap = 0.012; w = (1 - gap * (n - 1)) / n
    for i, (lab, col) in enumerate(chips):
        x = i * (w + gap)
        ax.add_patch(plt.Rectangle((x, 0.02), w, 0.085, transform=ax.transAxes,
                     facecolor=col, alpha=0.18, edgecolor=col, lw=1.4))
        ax.text(x + w / 2, 0.062, lab, transform=ax.transAxes, ha="center", va="center",
                fontsize=8.5, color="#1b2330", family="monospace")
    fig.suptitle("6 rows × 5 columns · 4 dtypes in one table  ·  mean score 67.14",
                 fontsize=12, fontweight="bold", color="#1b2330", y=0.995)
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, bbox_inches="tight", facecolor="white")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep06_indexing.py
"""
DS06 · Indexing & Selection — loc, iloc, boolean. Seeded, offline, reproduces to the digit.
Three doors into a DataFrame: .loc by label, .iloc by position, a boolean mask by condition.
"""
import pathlib
import numpy as np
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds06.png"


def build():
    rng = np.random.default_rng(7)          # the course contract: same seed → same numbers
    n = 12
    return pd.DataFrame({
        "user":     [f"u{i:02d}" for i in range(n)],
        "plan":     rng.choice(["free", "pro", "team"], size=n, p=[0.5, 0.3, 0.2]),
        "age":      rng.integers(18, 60, size=n),
        "sessions": rng.integers(0, 40, size=n),
        "spend":    np.round(rng.gamma(2.0, 25.0, size=n), 2),
    })


if __name__ == "__main__":
    df = build()
    print(df.to_string())

    loc_val = df.loc[3, "spend"]                          # Door 1: by LABEL
    print("\n.loc[3, 'spend'] (label)  =", loc_val)

    print("\n.iloc[0:3, 0:2] (position):")                # Door 2: by POSITION (half-open)
    print(df.iloc[0:3, 0:2].to_string())

    mask = (df["plan"] == "pro") & (df["spend"] > 40)     # Door 3: by CONDITION
    sel = df[mask]
    print("\nboolean (plan=='pro') & (spend>40):")
    print(sel.to_string())

    n_rows = len(sel)
    mean_spend = round(float(sel["spend"].mean()), 2)
    print("\nrows matched =", n_rows)
    print("mean spend   =", mean_spend)

    at_val = df.at[3, "spend"]                            # fast scalar accessor
    print(".at[3,'spend'] =", at_val, "  (== .loc?", at_val == loc_val, ")")

    df2 = build()                                          # reproduce
    assert df.equals(df2), "non-deterministic build!"
    assert n_rows == 2 and mean_spend == 45.34 and at_val == loc_val == 46.59
    print("\nreproduce: OK  | figure == terminal: OK")

    # ---- hero figure: full table with mask-matched rows lit pink + count badge ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    sel_idx = set(sel.index)
    cols = list(df.columns)
    fig, ax = plt.subplots(figsize=(8, 4.2)); ax.axis("off")
    cw = [0.13, 0.16, 0.16, 0.13, 0.22, 0.20]            # index + 5 cols
    cx = np.cumsum([0] + cw)
    rh = 0.066; top = 0.90
    headers = [""] + cols
    for k in range(6):
        ax.add_patch(plt.Rectangle((cx[k], top), cw[k], rh, transform=ax.transAxes,
                     facecolor="#FF4D9D" if k > 0 else "#FFFFFF", alpha=0.9 if k > 0 else 1, edgecolor="#cfd4da", lw=0.8))
        ax.text(cx[k] + cw[k] / 2, top + rh / 2, headers[k], transform=ax.transAxes, ha="center", va="center",
                fontsize=10.5, fontweight="bold", color="#FFFFFF" if k > 0 else "#7d8693", family="monospace")
    for r in range(len(df)):
        hot = r in sel_idx
        y = top - rh * (r + 1)
        vals = [str(r)] + [str(df.iloc[r][c]) for c in cols]
        for k in range(6):
            extra = (k == 5 and hot)                       # spend column on matched rows
            ax.add_patch(plt.Rectangle((cx[k], y), cw[k], rh, transform=ax.transAxes,
                         facecolor="#FF4D9D" if hot else "#FFFFFF",
                         alpha=(0.30 if extra else 0.16) if hot else 1.0,
                         edgecolor="#ececec", lw=0.6))
            ax.text(cx[k] + cw[k] / 2, y + rh / 2, vals[k], transform=ax.transAxes, ha="center", va="center",
                    fontsize=10, color="#8A8DA8" if (k == 0 or not hot) else "#1b2330",
                    fontweight="bold" if hot else "normal", family="monospace")
    # count badge
    ax.add_patch(plt.Rectangle((0.62, top + rh + 0.03), 0.36, 0.07, transform=ax.transAxes,
                 facecolor="#FF4D9D", alpha=0.92, edgecolor="none"))
    ax.text(0.80, top + rh + 0.065, "%d rows · mean spend %.2f" % (n_rows, mean_spend), transform=ax.transAxes,
            ha="center", va="center", fontsize=10.5, fontweight="bold", color="#FFFFFF", family="monospace")
    ax.text(0.02, 0.02, "mask = (plan=='pro') & (spend>40)", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=9, color="#8A8DA8", family="monospace")
    fig.suptitle("query, don't hunt  ·  2 of 12 rows  ·  mean spend 45.34",
                 fontsize=12.5, fontweight="bold", color="#1b2330", y=0.99)
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, bbox_inches="tight", facecolor="white")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep07_groupby.py
"""
DS07 · GroupBy — split, apply, combine. Seeded, offline, reproduces to the digit.
df.groupby(key).agg(...) splits rows by a key, applies an aggregation per group, combines into one frame.
"""
import pathlib
import numpy as np
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds07.png"


def make_df(seed=7, n=30):
    g = np.random.default_rng(seed)              # the course contract: same seed → same numbers
    cats = g.choice(["A", "B", "C"], size=n)     # category key
    value = np.round(g.normal(50, 10, size=n), 2)  # numeric feature
    return pd.DataFrame({"cat": cats, "value": value})


def summarize(df):
    s = df.groupby("cat").agg(mean=("value", "mean"), count=("value", "count"))
    s["mean"] = s["mean"].round(2)
    return s.sort_values("mean", ascending=False)     # rank by mean, highest first


if __name__ == "__main__":
    df = make_df()
    print("rows:", len(df))
    print(df.head(6).to_string(index=False)); print()

    summary = summarize(df)                        # SPLIT → APPLY → COMBINE → sort
    print(summary.to_string()); print()

    top_cat = summary.index[0]
    top_mean = float(summary["mean"].iloc[0])
    total = int(summary["count"].sum())
    print("top group: %s  mean=%.2f" % (top_cat, top_mean))
    print("rows accounted for:", total)

    pd.testing.assert_frame_equal(make_df(), df)   # reproduce: same seed → identical frame
    pd.testing.assert_frame_equal(summarize(make_df()), summary)
    assert total == 30
    fig_values = summary["mean"].to_dict()         # figure == terminal
    assert fig_values[top_cat] == top_mean
    print("reproduce: OK | figure==terminal: OK")

    # ---- hero figure: grouped bar chart of mean per group (sorted) ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#1b2330", "ytick.color": "#7d8693",
                         "grid.color": "#ececec"})
    cats = list(summary.index); means = list(summary["mean"]); counts = list(summary["count"])
    fig, ax = plt.subplots(figsize=(8, 3.6))
    bars = ax.bar(cats, means, width=0.62, color="#FF4D9D", edgecolor="#C8316F", lw=1.4, zorder=3)
    # winner top-edge accent (green) on the first bar
    ax.plot([bars[0].get_x(), bars[0].get_x() + bars[0].get_width()], [means[0], means[0]],
            color="#2BB57A", lw=4, zorder=5)
    ax.text(0, means[0] + 1.6, "▲ top", ha="center", color="#2BB57A", fontsize=10, fontweight="bold")
    for i, (m, c) in enumerate(zip(means, counts)):
        ax.text(i, m + 0.4, "%.2f" % m, ha="center", va="bottom", fontsize=12, fontweight="bold", color="#1b2330")
        ax.text(i, 2.0, "n=%d" % c, ha="center", va="bottom", fontsize=10, color="#10202b")
    ax.set_ylim(0, max(means) + 10); ax.set_ylabel("mean value")
    ax.grid(True, axis="y", lw=0.7); ax.set_axisbelow(True)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.set_title("group %s leads — mean %.2f across %d rows  ·  one line of groupby" % (top_cat, top_mean, total),
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep08_merge.py
"""
DS08 · Merge, Join & Reshape — pivot, melt. Seeded, offline, reproduces to the digit.
merge aligns two tables by a shared key (inner vs left decides who survives); pivot_table reshapes long→wide, melt reverses.
"""
import pathlib
import numpy as np
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds08.png"
N = 12


def build_orders(seed=7):
    r = np.random.default_rng(seed)              # the course contract: same seed → same numbers
    return pd.DataFrame({
        "oid": np.arange(1, N + 1),
        "cust": r.integers(1, 6, size=N),         # ids 1..5; 4 & 5 are NOT in the lookup
        "status": r.choice(["paid", "refund"], size=N, p=[0.7, 0.3]),
    })


CUSTOMERS = pd.DataFrame({"cust": [1, 2, 3], "region": ["north", "south", "north"]})  # intentionally incomplete


def pivot_of(orders):
    return (orders.merge(CUSTOMERS, on="cust", how="left")
                  .pivot_table(index="region", columns="status", values="oid", aggfunc="count", fill_value=0))


if __name__ == "__main__":
    orders = build_orders()
    m_inner = orders.merge(CUSTOMERS, on="cust", how="inner")
    m_left = orders.merge(CUSTOMERS, on="cust", how="left")
    n_inner, n_left = len(m_inner), len(m_left)
    n_dropped = n_left - n_inner
    n_unmatched = int(m_left["region"].isna().sum())

    pivot = pivot_of(orders)
    pivot_total = int(pivot.to_numpy().sum())
    long_again = pivot.reset_index().melt(id_vars="region", var_name="status", value_name="count")
    melt_total = int(long_again["count"].sum())

    assert pivot_of(build_orders()).equals(pivot), "not reproducible"
    assert pivot_total == n_inner and melt_total == pivot_total and n_dropped == n_unmatched

    print("inner rows :", n_inner)
    print("left  rows :", n_left)
    print("dropped by inner (unmatched keys):", n_dropped)
    print("\npivot_table (region x status, count of orders):"); print(pivot)
    print("\npivot grand total :", pivot_total)
    print("melt round-trip total :", melt_total)
    print("reproduce check: PASS")

    # ---- hero figure: pivot table + inner/left bars on a white card ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cols = list(pivot.columns); rows = list(pivot.index)
    fig, ax = plt.subplots(figsize=(8, 3.6)); ax.axis("off")
    # pivot table (left half)
    cw = 0.16; x0 = 0.04; y0 = 0.66; rh = 0.17
    ax.text(x0, y0 + 2 * rh + 0.04, "pivot: region × status → order count", fontsize=10, color="#7d8693", family="monospace")
    for j, c in enumerate(cols):
        ax.add_patch(plt.Rectangle((x0 + (j + 1) * cw, y0 + rh), cw, rh, transform=ax.transAxes, facecolor="#FF4D9D", alpha=0.9, edgecolor="#cfd4da", lw=0.8))
        ax.text(x0 + (j + 1.5) * cw, y0 + rh * 1.5, c, ha="center", va="center", transform=ax.transAxes, fontsize=11, fontweight="bold", color="#fff", family="monospace")
    for i, r in enumerate(rows):
        ax.add_patch(plt.Rectangle((x0, y0 - i * rh), cw, rh, transform=ax.transAxes, facecolor="#FF4D9D", alpha=0.9, edgecolor="#cfd4da", lw=0.8))
        ax.text(x0 + cw / 2, y0 - i * rh + rh / 2, r, ha="center", va="center", transform=ax.transAxes, fontsize=11, fontweight="bold", color="#fff", family="monospace")
        for j, c in enumerate(cols):
            ax.add_patch(plt.Rectangle((x0 + (j + 1) * cw, y0 - i * rh), cw, rh, transform=ax.transAxes, facecolor="#FFFFFF", edgecolor="#ececec", lw=0.8))
            ax.text(x0 + (j + 1.5) * cw, y0 - i * rh + rh / 2, str(pivot.loc[r, c]), ha="center", va="center", transform=ax.transAxes, fontsize=14, fontweight="bold", color="#1b2330", family="monospace")
    ax.text(x0, y0 - len(rows) * rh - 0.02, "grand total = %d  (= inner rows)" % pivot_total, transform=ax.transAxes, fontsize=10, color="#2BB57A", fontweight="bold", family="monospace")
    # inner vs left bars (right half)
    bx = 0.66; bw = 0.30
    ax.text(bx, 0.92, "rows kept by join", transform=ax.transAxes, fontsize=10.5, color="#7d8693", family="monospace")
    ax.add_patch(plt.Rectangle((bx, 0.66), bw * n_inner / n_left, 0.13, transform=ax.transAxes, facecolor="#2BD9FF", alpha=0.85, edgecolor="none"))
    ax.text(bx + bw * n_inner / n_left + 0.01, 0.725, "inner = %d" % n_inner, transform=ax.transAxes, fontsize=11, va="center", color="#1b2330", fontweight="bold", family="monospace")
    ax.add_patch(plt.Rectangle((bx, 0.46), bw, 0.13, transform=ax.transAxes, facecolor="#FF4D9D", alpha=0.85, edgecolor="none"))
    ax.text(bx + bw + 0.01, 0.525, "left = %d" % n_left, transform=ax.transAxes, fontsize=11, va="center", color="#1b2330", fontweight="bold", family="monospace")
    ax.annotate("", xy=(bx + bw, 0.40), xytext=(bx + bw * n_inner / n_left, 0.40), transform=ax.transAxes,
                arrowprops=dict(arrowstyle="<->", color="#8A8DA8", lw=1.3))
    ax.text(bx + bw * (n_inner / n_left + 1) / 2, 0.34, "%d dropped" % n_dropped, transform=ax.transAxes, ha="center", fontsize=10, color="#FF5C6C", fontweight="bold", family="monospace")
    fig.suptitle("left-merge → pivot  ·  %d matched orders (%d dropped by inner)" % (pivot_total, n_dropped),
                 fontsize=12.5, fontweight="bold", color="#1b2330", y=0.99)
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep09_features.py
"""
DS09 · Missing Data & Feature Engineering. Seeded, offline, reproduces to the digit.
The fixed recipe: impute (median, robust) → encode (get_dummies) → scale (standardize). Clean first, then engineer.
"""
import pathlib
import numpy as np
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds09.png"
N = 200


def build():
    r = np.random.default_rng(7)                 # the course contract: same seed → same numbers
    df = pd.DataFrame({
        "age": r.integers(18, 70, size=N).astype(float),
        "income": r.normal(60000, 15000, size=N).round(0),
        "tenure": r.integers(0, 120, size=N).astype(float),
        "region": r.choice(["north", "south", "east"], size=N),
    })
    for col, frac in [("age", 0.08), ("income", 0.12), ("tenure", 0.05)]:
        k = int(N * frac)
        idx = r.choice(N, size=k, replace=False)
        df.loc[idx, col] = np.nan
    return df


if __name__ == "__main__":
    df = build()
    na_counts = df.isna().sum()
    print("NaNs per column:"); print(na_counts.to_string())

    medians = df[["age", "income", "tenure"]].median()
    print("\nMedians used to impute:"); print(medians.round(2).to_string())

    df_imp = df.copy()
    df_imp[["age", "income", "tenure"]] = df_imp[["age", "income", "tenure"]].fillna(medians)
    assert df_imp.isna().sum().sum() == 0          # 2. no holes left

    df_enc = pd.get_dummies(df_imp, columns=["region"], prefix="region")   # 3. one-hot
    print("\nColumns after one-hot:"); print(list(df_enc.columns))

    mu = df_enc["income"].mean(); sd = df_enc["income"].std(ddof=0)        # 4. standardize
    df_enc["income_z"] = (df_enc["income"] - mu) / sd
    zmin, zmax = float(df_enc["income_z"].min()), float(df_enc["income_z"].max())
    print("\nincome   mean=%.2f  std=%.2f" % (mu, sd))
    print("income_z mean=%.4f  std=%.4f" % (df_enc["income_z"].mean(), df_enc["income_z"].std(ddof=0)))
    print("income_z min=%.3f  max=%.3f" % (zmin, zmax))

    assert build().isna().sum().equals(na_counts)  # reproduce: same holes
    assert list(na_counts.values) == [16, 24, 10, 0]
    holes = int(na_counts.sum())
    print("\nreproduce check: PASS (NaN counts identical on rerun)")

    # ---- hero figure: missingness bars + income → income_z ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#1b2330", "ytick.color": "#7d8693", "grid.color": "#ececec"})
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(8, 3.5), gridspec_kw={"width_ratios": [1, 1.05]})
    # left: missingness bars
    cols = ["age", "income", "tenure", "region"]; vals = [int(na_counts[c]) for c in cols]
    axL.bar(cols, vals, color="#FF4D9D", edgecolor="#C8316F", lw=1.2, zorder=3)
    for i, v in enumerate(vals):
        axL.text(i, v + 0.4, str(v), ha="center", va="bottom", fontsize=11, fontweight="bold", color="#1b2330")
    axL.set_ylim(0, 27); axL.set_ylabel("NaN count"); axL.set_title("1 · missingness  (impute)", fontsize=11)
    axL.grid(True, axis="y", lw=0.7); axL.set_axisbelow(True)
    for s in ["top", "right"]:
        axL.spines[s].set_visible(False)
    # right: income raw vs income_z (strip on shared 0-centered? two rows)
    inc = df_enc["income"].to_numpy(); z = df_enc["income_z"].to_numpy()
    axR.scatter(inc / 1000, np.full(N, 1.0), s=10, color="#2BD9FF", alpha=0.5)
    axR.axvline(mu / 1000, color="#2BB57A", lw=1.5, ymin=0.55, ymax=0.95)
    axR.text(mu / 1000, 1.18, "income mean %.0f" % mu, ha="center", fontsize=8.5, color="#1b2330")
    # map z to a second row, but on its own scale annotation
    z_disp = z * 6 + 58   # visually place z near the same horizontal band (cosmetic only; labels carry truth)
    axR.scatter(z_disp, np.full(N, 0.0), s=10, color="#4DE0A0", alpha=0.6)
    axR.axvline(0 * 6 + 58, color="#2BB57A", lw=1.5, ymin=0.05, ymax=0.45)
    axR.text(58, -0.32, "income_z  mean 0 · std 1  (min %.2f · max %.2f)" % (zmin, zmax), ha="center", fontsize=8.5, color="#1b2330")
    axR.annotate("(x − μ) / σ", xy=(58, 0.5), ha="center", fontsize=10, color="#7d8693")
    axR.set_yticks([0, 1]); axR.set_yticklabels(["income_z", "income (k)"], fontsize=9)
    axR.set_ylim(-0.6, 1.4); axR.set_title("4 · scale to one footing", fontsize=11)
    for s in ["top", "right"]:
        axR.spines[s].set_visible(False)
    fig.suptitle("%d holes filled · region → 3 columns · income → mean 0, std 1" % holes,
                 fontsize=12, fontweight="bold", color="#1b2330", y=1.0)
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white")
    print("saved:", FIG.name, "| holes:", holes)


In [ ]:
%%writefile ep10_anatomy.py
"""
DS10 · Anatomy of a Figure — fig, axes, artists. Seeded, offline, reproduces to the digit.
Matplotlib is an object hierarchy: a Figure holds Axes; every visible thing is an artist you own a handle to.
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds10.png"


def build_data():
    rng = np.random.default_rng(7)               # the course contract: same seed → same digits
    x = np.linspace(0, 10, 12)                    # 12 evenly spaced points
    noise = rng.normal(0, 1.0, size=x.size)
    y = 2.0 * x + 1.0 + noise                     # true line y = 2x + 1, perturbed
    return x, y


def make_figure(x, y):
    fig, ax = plt.subplots(figsize=(8, 4.6))      # ONE call → BOTH objects
    (line,) = ax.plot(x, y, marker="o", color="#2BD9FF", label="data")   # line is an artist
    slope, intercept = np.polyfit(x, y, 1)
    ax.plot(x, slope * x + intercept, color="#5FBF1F", lw=2.4, label="lstsq fit")
    ax.set_title("anatomy of a figure", fontsize=13, fontweight="bold")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    leg = ax.legend(loc="upper left")
    i = int(y.argmax()); peak_x, peak_y = float(x[i]), float(y[i])
    for s in ax.spines.values():
        s.set_color("#7d8693")
    n_tick = len(ax.get_xticklabels())
    return fig, ax, line, leg, slope, intercept, peak_x, peak_y, n_tick


if __name__ == "__main__":
    x, y = build_data()
    fig, ax, line, leg, slope, intercept, peak_x, peak_y, n_tick = make_figure(x, y)
    n_points = len(line.get_xdata())

    # ---- name every part with real matplotlib annotations (the anatomy) ----
    lab = dict(fontsize=9, color="#1b2330", fontfamily="monospace",
               arrowprops=dict(arrowstyle="->", color="#8A8DA8", lw=1.1))
    ax.annotate("line (Line2D artist)", xy=(x[6], y[6]), xytext=(2.0, 19.5), **lab)
    ax.annotate("lstsq fit", xy=(7.2, slope * 7.2 + intercept), xytext=(7.4, 7.0), **lab)
    ax.annotate("peak %.3f @ x=10" % peak_y, xy=(peak_x, peak_y), xytext=(5.2, 22.0),
                fontsize=9, color="#C8631F", fontfamily="monospace",
                arrowprops=dict(arrowstyle="->", color="#FF9F40", lw=1.4))
    ax.annotate("legend (artist)", xy=(0.5, 19.0), xytext=(0.4, 13.0), **lab)
    ax.annotate("tick labels (artists)", xy=(8.0, 0.0), xytext=(5.6, -3.2), **lab)
    ax.annotate("title (artist)", xy=(5.0, 23.2), xytext=(0.2, 24.2), fontsize=9,
                color="#1b2330", fontfamily="monospace",
                arrowprops=dict(arrowstyle="->", color="#8A8DA8", lw=1.1))
    ax.set_ylim(-5, 26)

    # ---- containment: dashed Figure border + Axes border, in figure coords ----
    fig.canvas.draw()
    fig.add_artist(Rectangle((0.01, 0.01), 0.98, 0.98, transform=fig.transFigure,
                   fill=False, ec="#9EFF3D", lw=1.6, ls="--"))
    fig.text(0.025, 0.955, "Figure", color="#5FBF1F", fontsize=11, fontweight="bold", fontfamily="monospace")
    bb = ax.get_position()
    fig.add_artist(Rectangle((bb.x0 - 0.005, bb.y0 - 0.005), bb.width + 0.01, bb.height + 0.01,
                   transform=fig.transFigure, fill=False, ec="#2BD9FF", lw=1.2, ls=":"))
    fig.text(bb.x0, bb.y1 + 0.005, "Axes", color="#1f8fb0", fontsize=10, fontweight="bold", fontfamily="monospace")

    print("figure objects : fig=%s  ax=%s" % (type(fig).__name__, type(ax).__name__))
    print("line artist    : %s, n_points=%d" % (type(line).__name__, n_points))
    print("legend entries : %d" % len(leg.get_texts()))
    print("peak y         : %.3f at x=%.3f" % (peak_y, peak_x))
    print("lstsq fit      : slope=%.3f intercept=%.3f" % (slope, intercept))

    assert abs(float(y[int(y.argmax())]) - peak_y) < 1e-9       # figure == terminal
    assert np.array_equal(build_data()[1], y)                    # reproducible
    assert n_points == 12 and round(peak_y, 3) == 21.357 and round(slope, 3) == 2.045
    print("reproduce check: (12, 21.357, 2.045, 0.678)  (run twice → identical)")

    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep11_eda.py
"""
DS11 · EDA Plots — scatter, hist, box for understanding data. Seeded, offline, reproduces to the digit.
Before you model, you look: a scatter colored by class + marginal histograms reveal which feature separates.
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds11.png"
N = 150


def build():
    rng = np.random.default_rng(7)                  # the course contract: same seed → same numbers
    x0 = rng.normal(2.0, 0.8, N); y0 = rng.normal(2.0, 0.8, N)   # class 0
    x1 = rng.normal(4.0, 1.0, N); y1 = rng.normal(4.5, 1.0, N)   # class 1
    X = np.concatenate([x0, x1]); Y = np.concatenate([y0, y1])
    label = np.concatenate([np.zeros(N, int), np.ones(N, int)])
    return X, Y, label


if __name__ == "__main__":
    X, Y, label = build()
    m0, m1 = label == 0, label == 1
    c0_f1, c0_f2 = X[m0].mean(), Y[m0].mean()
    c1_f1, c1_f2 = X[m1].mean(), Y[m1].mean()
    gap_f1, gap_f2 = abs(c1_f1 - c0_f1), abs(c1_f2 - c0_f2)
    better = "feature2" if gap_f2 > gap_f1 else "feature1"

    print("class0  feature1 mean: %.3f  feature2 mean: %.3f" % (c0_f1, c0_f2))
    print("class1  feature1 mean: %.3f  feature2 mean: %.3f" % (c1_f1, c1_f2))
    print("feature2 mean gap (class1 - class0): %.3f" % gap_f2)
    print("feature1 gap: %.3f   feature2 gap: %.3f" % (gap_f1, gap_f2))
    print("better separator: %s" % better)

    CYAN, PINK, LIME = "#2BD9FF", "#FF4D9D", "#5FBF1F"
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig = plt.figure(figsize=(7, 5.4))
    gs = fig.add_gridspec(2, 2, width_ratios=(4, 1), height_ratios=(1, 4), wspace=0.05, hspace=0.05)
    ax = fig.add_subplot(gs[1, 0]); axx = fig.add_subplot(gs[0, 0], sharex=ax); axy = fig.add_subplot(gs[1, 1], sharey=ax)
    ax.scatter(X[m0], Y[m0], c=CYAN, s=16, alpha=0.7, label="class 0", edgecolor="none")
    ax.scatter(X[m1], Y[m1], c=PINK, s=16, alpha=0.7, label="class 1", edgecolor="none")
    ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2"); ax.legend(loc="upper left", fontsize=9)
    bins = np.linspace(-1, 8, 26)
    axx.hist(X[m0], bins=bins, color=CYAN, alpha=0.6); axx.hist(X[m1], bins=bins, color=PINK, alpha=0.6)
    axy.hist(Y[m0], bins=bins, color=CYAN, alpha=0.6, orientation="horizontal")
    axy.hist(Y[m1], bins=bins, color=PINK, alpha=0.6, orientation="horizontal")
    axx.tick_params(labelbottom=False, length=0); axy.tick_params(labelleft=False, length=0)
    for a in (axx, axy):
        for s in a.spines.values():
            s.set_visible(False)
    fig_gap = round(c1_f2 - c0_f2, 3)
    xm = axy.get_xlim()[1]
    axy.annotate("", xy=(xm * 0.5, c1_f2), xytext=(xm * 0.5, c0_f2), arrowprops=dict(arrowstyle="<->", color=LIME, lw=2.4))
    axy.text(xm * 0.54, (c0_f2 + c1_f2) / 2, "gap\n%.3f" % fig_gap, color="#1b2330", fontsize=9, va="center", fontweight="bold")
    fig.suptitle("plot before you model · feature 2 separates · gap %.3f" % fig_gap, fontsize=12, fontweight="bold", color="#1b2330")

    Xb, Yb, lb = build()
    assert np.array_equal(X, Xb) and np.array_equal(Y, Yb) and np.array_equal(label, lb)
    assert round(fig_gap, 3) == round(gap_f2, 3)
    print("reproduce: OK   figure_gap == terminal_gap: %.3f" % round(fig_gap, 3))

    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep12_report.py
"""
DS12 · Visualizing Model Results — confusion matrix, ROC/PR. Seeded, offline, reproduces to the digit.
Accuracy is one number hiding four (TN/FP/FN/TP); the matrix + ROC are the model's real report card. No sklearn.
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds12.png"
N = 200


def build():
    rng = np.random.default_rng(7)                       # the course contract: same seed → same numbers
    y_true = rng.integers(0, 2, size=N)
    noise = rng.normal(0, 0.35, size=N)
    y_score = np.clip(0.5 + (y_true - 0.5) * 0.7 + noise, 0, 1)   # leans right, not perfect
    return y_true, y_score


if __name__ == "__main__":
    y_true, y_score = build()
    y_pred = (y_score >= 0.5).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_true == 1))); tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0))); fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    cm = np.array([[tn, fp], [fn, tp]])
    acc = (tp + tn) / N; prec = tp / (tp + fp); rec = tp / (tp + fn)

    P, Ng = int((y_true == 1).sum()), int((y_true == 0).sum())
    order = np.argsort(-y_score); ys = y_true[order]
    tpr = np.concatenate([[0], np.cumsum(ys) / P]); fpr = np.concatenate([[0], np.cumsum(1 - ys) / Ng])
    auc = float(np.trapezoid(tpr, fpr))

    print("class balance:", Ng, "neg,", P, "pos")
    print("confusion matrix [[TN,FP],[FN,TP]]:"); print(cm)
    print("accuracy : %.3f" % acc); print("precision: %.3f" % prec)
    print("recall   : %.3f" % rec); print("ROC-AUC  : %.3f" % auc)

    rng2 = np.random.default_rng(7)
    assert np.array_equal(y_true, rng2.integers(0, 2, size=N)), "seed drift!"
    assert cm.tolist() == [[84, 11], [20, 85]] and round(auc, 3) == 0.927
    print("reproduces:", True)

    # ---- hero figure: confusion matrix heatmap + ROC curve ----
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig, (axc, axr) = plt.subplots(1, 2, figsize=(8.4, 4.0), gridspec_kw={"width_ratios": [1, 1.05]})
    # confusion matrix
    axc.imshow(cm, cmap="Blues")
    axc.set_xticks([0, 1], ["pred 0", "pred 1"]); axc.set_yticks([0, 1], ["true 0", "true 1"])
    lab = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            col = "white" if cm[i, j] > cm.max() * 0.6 else "#1b2330"
            axc.text(j, i, "%s\n%d" % (lab[i][j], cm[i, j]), ha="center", va="center", color=col, fontsize=16, fontweight="bold")
    axc.set_title("confusion matrix · acc %.3f" % acc, fontsize=12, fontweight="bold")
    # ROC
    axr.plot(fpr, tpr, color="#5FBF1F", lw=2.6, label="ROC (AUC %.3f)" % auc)
    axr.fill_between(fpr, tpr, color="#9EFF3D", alpha=0.15)
    axr.plot([0, 1], [0, 1], color="#8A8DA8", lw=1.2, ls="--", label="random")
    axr.scatter([fp / Ng], [tp / P], color="#FF4D9D", s=60, zorder=5, label="thr=0.5")
    axr.set_xlabel("FPR"); axr.set_ylabel("TPR"); axr.set_xlim(0, 1); axr.set_ylim(0, 1.02)
    axr.legend(loc="lower right", fontsize=9); axr.set_title("ROC curve", fontsize=12, fontweight="bold")
    for s in ["top", "right"]:
        axr.spines[s].set_visible(False)
    fig.suptitle("84.5% accurate — but recall 0.810 misses 1 in 5 positives", fontsize=12.5, fontweight="bold", color="#1b2330", y=1.02)
    fig.tight_layout()

    assert int(cm[1, 1]) == tp
    print("figure TP == terminal TP:", int(cm[1, 1]))
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep13_dashboard.py
"""
DS13 · Subplots & Styling for Reports. Seeded, offline, reproduces to the digit.
plt.subplots(2,2) → one styled, deterministic dashboard. Four panels on one canvas beats four orphan PNGs.
"""
import pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds13.png"
TMP = FIG.parent / "_ds13_repro.png"


def build_data():
    rng = np.random.default_rng(7)                       # the course contract: same seed → same numbers
    vals = rng.normal(50, 10, 200)
    x = np.linspace(0, 10, 40); y = 2.0 * x + 1.0 + rng.normal(0, 2.0, 40)
    cats = ["A", "B", "C", "D"]; heights = rng.integers(10, 60, 4)
    cum = np.cumsum(rng.integers(1, 8, 12))
    return vals, x, y, cats, heights, cum


def stats(vals, x, y, cats, heights, cum):
    slope, intercept = np.polyfit(x, y, 1)
    ti = int(np.argmax(heights))
    return {"hist_mean": round(float(vals.mean()), 2), "fit_slope": round(float(slope), 2),
            "fit_intercept": round(float(intercept), 2), "top_cat": cats[ti],
            "top_height": int(heights[ti]), "cum_end": int(cum[-1])}


def make_figure(vals, x, y, cats, heights, cum, s, fname):
    plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.color": "#ececec",
                         "lines.linewidth": 2.0, "axes.edgecolor": "#7d8693", "figure.facecolor": "white",
                         "text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig, axes = plt.subplots(2, 2, figsize=(8, 6))
    ax_hist, ax_scat, ax_bar, ax_cum = axes.ravel()
    ax_hist.hist(vals, bins=15, color="#9EFF3D", edgecolor="#1b2330")
    ax_hist.set_title("distribution"); ax_hist.set_xlabel("value"); ax_hist.set_ylabel("count")
    ax_scat.scatter(x, y, s=18, color="#2BD9FF", edgecolor="#1b2330")
    ax_scat.plot(x, s["fit_slope"] * x + s["fit_intercept"], color="#FF4D9D")
    ax_scat.annotate("slope=%.2f" % s["fit_slope"], xy=(0.05, 0.88), xycoords="axes fraction", color="#1b2330", fontweight="bold")
    ax_scat.set_title("fit"); ax_scat.set_xlabel("x"); ax_scat.set_ylabel("y")
    ax_bar.bar(cats, heights, color="#FF9F40", edgecolor="#1b2330")
    ax_bar.set_title("categories"); ax_bar.set_xlabel("group"); ax_bar.set_ylabel("height")
    ax_cum.plot(range(1, len(cum) + 1), cum, color="#4DE0A0", marker="o")
    ax_cum.set_title("cumulative"); ax_cum.set_xlabel("step"); ax_cum.set_ylabel("total")
    fig.suptitle("seeded report dashboard", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(fname, dpi=130)
    ann = [c for c in ax_scat.texts if "slope=" in c.get_text()][0]
    fig_slope = float(ann.get_text().split("=")[1])
    plt.close(fig)
    return fig_slope


if __name__ == "__main__":
    d = build_data(); s = stats(*d)
    print("PANEL 1  hist   mean      =", s["hist_mean"])
    print("PANEL 2  fit    slope     =", s["fit_slope"])
    print("PANEL 3  bars   top       =", s["top_cat"], "@", s["top_height"])
    print("PANEL 4  cum    endpoint  =", s["cum_end"])

    fig_slope = make_figure(*d, s, str(FIG))
    print("FIGURE   annotated slope  =", round(fig_slope, 2))
    assert round(fig_slope, 2) == s["fit_slope"], "figure != terminal"
    print("CHECK    figure == terminal:", round(fig_slope, 2) == s["fit_slope"])

    d2 = build_data(); s2 = stats(*d2)
    make_figure(*d2, s2, str(TMP))
    a, b = plt.imread(str(FIG)), plt.imread(str(TMP))
    print("CHECK    image bytes identical:", np.array_equal(a, b))
    print("CHECK    stats identical      :", s == s2)
    TMP.unlink(missing_ok=True)
    print("saved:", FIG.name)


In [ ]:
%%writefile ep14_image_arrays.py
"""
DS14 · Images Are NumPy Arrays — shape, dtype, color spaces. Seeded, offline, reproduces to the digit.
An image is an (H,W,C) uint8 array; cv2 operates on those numbers. OpenCV stores color as BGR, not RGB.
"""
import pathlib
import numpy as np
import cv2

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds14.png"
H = W = 64


def build_image(seed=7):
    """A structured synthetic BGR uint8 image — gradients + cv2 shapes + seeded noise. Pure code, deterministic."""
    rng = np.random.default_rng(seed)
    img = np.zeros((H, W, 3), np.uint8)
    img[:, :, 0] = np.tile(np.linspace(40, 200, W).astype(np.uint8), (H, 1))      # B: horizontal gradient
    img[:, :, 1] = 165                                                            # G: bright base (luminance-heavy)
    img[:, :, 2] = np.tile(np.linspace(30, 150, H).astype(np.uint8)[:, None], (1, W))  # R: vertical gradient
    cv2.circle(img, (18, 18), 12, (255, 255, 0), -1)                              # cyan disc (B+G), BGR
    cv2.rectangle(img, (40, 40), (58, 58), (0, 0, 255), -1)                       # red square, BGR
    noise = rng.integers(-12, 13, size=(H, W, 3))
    return np.clip(img.astype(int) + noise, 0, 255).astype(np.uint8)


if __name__ == "__main__":
    img = build_image()
    print("shape:", img.shape)              # (64, 64, 3)
    print("dtype:", img.dtype)              # uint8
    print("min/max:", int(img.min()), int(img.max()))

    b, g, r = cv2.split(img)                # OpenCV order = B, G, R
    b_mean = round(float(b.mean()), 2); g_mean = round(float(g.mean()), 2); r_mean = round(float(r.mean()), 2)
    print("B mean:", b_mean); print("G mean:", g_mean); print("R mean:", r_mean)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # (H, W)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)     # (H, W, 3)
    gray_mean = round(float(gray.mean()), 2)
    print("gray shape:", gray.shape); print("hsv shape:", hsv.shape)
    print("gray mean:", gray_mean)
    plain_avg = round((b_mean + g_mean + r_mean) / 3, 2)
    print("plain avg:", plain_avg, "(gray leans toward G:", gray_mean > plain_avg, ")")

    assert np.array_equal(img, build_image()), "image not reproducible"
    assert img.shape == (64, 64, 3) and img.dtype == np.uint8
    print("OK reproduce + figure==terminal")

    # ---- hero figure: RGB image + B/G/R channel heatmaps ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # honest colors for display
    fig, ax = plt.subplots(1, 4, figsize=(8.6, 2.7))
    ax[0].imshow(rgb); ax[0].set_title("image (64,64,3) uint8", fontsize=10)
    chans = [("B", b, b_mean, "Blues"), ("G", g, g_mean, "Greens"), ("R", r, r_mean, "Reds")]
    for k, (nm, ch, mn, cmap) in enumerate(chans):
        im = ax[k + 1].imshow(ch, cmap=cmap, vmin=0, vmax=255)
        ax[k + 1].set_title(nm, fontsize=11, fontweight="bold")
        ax[k + 1].set_xlabel("mean = %.2f" % mn, fontsize=9)
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle("an image is just an (H,W,3) uint8 array — gray %.2f leans toward G (avg %.2f)" % (gray_mean, plain_avg),
                 fontsize=11.5, fontweight="bold", color="#1b2330", y=1.04)
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep15_preprocess.py
"""
DS15 · Preprocessing for ML — resize, crop, normalize, dtype. Seeded, offline, reproduces to the digit.
Models eat fixed-size float32 [0,1] tensors: resize → crop ROI → cast uint8→float32 /255. The pixels→network contract.
"""
import pathlib
import numpy as np
import cv2

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds15.png"


def build_pipeline():
    rng = np.random.default_rng(7)                                  # the course contract: same seed → same numbers
    img = rng.integers(0, 256, size=(200, 300, 3), dtype=np.uint8)  # synthetic BGR, 200×300
    img[80:130, 120:180] = 255   # solid white block (centered → survives INTER_AREA + crop → max stays 255 → 1.0)
    img[80:130, 60:110] = 0      # solid black block (→ min stays 0)
    resized = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)  # 1) fixed model size
    cropped = resized[16:112, 16:112]                              # 2) center ROI, plain numpy → 96×96
    normed = cropped.astype(np.float32) / 255.0                    # 3) cast + scale → [0,1]
    return img, resized, cropped, normed


def stat(name, a):
    return (name, tuple(a.shape), str(a.dtype), float(a.min()), float(a.max()), float(a.mean()))


if __name__ == "__main__":
    img, resized, cropped, normed = build_pipeline()
    rows = [stat("original", img), stat("resized", resized), stat("cropped", cropped), stat("normalized", normed)]

    print("%-11s %-15s %-9s %7s %7s %8s" % ("stage", "shape", "dtype", "min", "max", "mean"))
    for nm, shp, dt, mn, mx, me in rows:
        print("%-11s %-15s %-9s %7.3f %7.3f %8.3f" % (nm, str(shp), dt, mn, mx, me))

    img2, resized2, cropped2, normed2 = build_pipeline()
    assert np.array_equal(normed, normed2)
    print("reproduce check: identical on re-run -> OK")

    fig_norm_max = round(float(normed.max()), 3); fig_norm_mean = round(float(normed.mean()), 3)
    assert fig_norm_max == round(rows[3][4], 3) and fig_norm_mean == round(rows[3][5], 3)
    assert img.shape == (200, 300, 3) and resized.shape == (128, 128, 3) and cropped.shape == (96, 96, 3)
    assert str(normed.dtype) == "float32" and fig_norm_max == 1.0
    print("figure values: norm_max=%s  norm_mean=%s" % (fig_norm_max, fig_norm_mean))

    # ---- hero figure: original vs resized + before/after stats table ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#FF9F40", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig = plt.figure(figsize=(8.4, 4.6))
    gs = fig.add_gridspec(2, 2, height_ratios=[2.1, 1.5], hspace=0.5, wspace=0.25)
    axo = fig.add_subplot(gs[0, 0]); axr = fig.add_subplot(gs[0, 1]); axt = fig.add_subplot(gs[1, :]); axt.axis("off")
    axo.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axo.set_title("original 200×300", fontsize=10)
    axr.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)); axr.set_title("resized 128×128", fontsize=10)
    for a in (axo, axr):
        for s in a.spines.values():
            s.set_color("#FF9F40"); s.set_linewidth(2)
    # table
    cols = ["stage", "shape", "dtype", "max", "mean"]
    tbl = [[nm, str(shp), dt, "%.1f" % mx, "%.3f" % me] for nm, shp, dt, mn, mx, me in rows]
    t = axt.table(cellText=tbl, colLabels=cols, cellLoc="center", loc="center")
    t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1, 1.6)
    for j in range(len(cols)):
        t[0, j].set_facecolor("#FF4D9D"); t[0, j].set_text_props(color="white", fontweight="bold")
    t[4, 2].set_facecolor("#D6F0FF")        # float32 dtype cell (cyan tint)
    t[4, 3].set_facecolor("#D6FBE9")        # max 1.0 cell (green tint)
    fig.suptitle("fixed size · float32 · [0,1]  —  max 1.0, mean %.3f · preprocessing is non-optional" % fig_norm_mean,
                 fontsize=11.5, fontweight="bold", color="#1b2330", y=1.0)
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep16_edges.py
"""
DS16 · Filtering & Edges — blur, Sobel, Canny. Seeded, offline, reproduces to the digit.
An edge is where brightness changes fast (a gradient). Blur (denoise) → Sobel (gradient) → Canny (thin+threshold).
"""
import pathlib
import numpy as np
import cv2

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds16.png"


def make_shape_image():
    img = np.full((200, 200), 60, dtype=np.uint8)        # gray background = 60
    cv2.rectangle(img, (30, 40), (120, 150), 255, -1)    # filled white rectangle
    cv2.circle(img, (150, 60), 35, 255, -1)              # filled white circle
    return img


def pipeline(img):
    blurred = cv2.GaussianBlur(img, (5, 5), 0)           # 1) denoise
    edges = cv2.Canny(blurred, 50, 150)                  # 2) gradient + thin + hysteresis
    return blurred, edges


if __name__ == "__main__":
    img = make_shape_image()
    print("image shape :", img.shape)
    print("image dtype :", img.dtype)
    print("input values:", np.unique(img).tolist())

    blurred, edges = pipeline(img)
    print("blur kernel : (5, 5)"); print("canny thresh: (50, 150)")
    print("edge values :", np.unique(edges).tolist())
    edge_px = int((edges == 255).sum())
    print("edge pixels :", edge_px)

    gx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    on = round(float(mag[edges == 255].mean()), 1); off = round(float(mag[edges == 0].mean()), 1)
    print("grad@edge   :", on); print("grad@flat   :", off)
    assert on > 10 * off                                 # edges = big gradients

    _, edges2 = pipeline(make_shape_image())
    assert np.array_equal(edges, edges2), "non-deterministic!"
    print("reproduce   : identical (edges == edges2)")
    print("figure value:", edge_px, "== terminal edge pixels  OK")

    # ---- hero figure: original · Sobel magnitude · Canny edges ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#FF9F40", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig, ax = plt.subplots(1, 3, figsize=(8.6, 3.1))
    ax[0].imshow(img, cmap="gray", vmin=0, vmax=255); ax[0].set_title("original (uint8) · bg 60 / shape 255", fontsize=9.5)
    ax[1].imshow(mag, cmap="inferno"); ax[1].set_title("Sobel |∇| · high on the rim", fontsize=9.5)
    # Canny: black bg, orange edges
    edge_cmap = ListedColormap(["#0b0b12", "#FF9F40"])
    ax[2].imshow((edges == 255).astype(int), cmap=edge_cmap, vmin=0, vmax=1)
    ax[2].set_title("Canny edges ∈ {0,255}", fontsize=9.5)
    ax[2].text(0.5, -0.16, "edge pixels = %d" % edge_px, transform=ax[2].transAxes, ha="center",
               fontsize=11, fontweight="bold", color="#1b2330",
               bbox=dict(boxstyle="round", fc="#FFE8CC", ec="#FF9F40"))
    for a in ax:
        a.set_xticks([0, 100, 200]); a.set_yticks([0, 100, 200])
        for s in a.spines.values():
            s.set_color("#FF9F40")
    fig.suptitle("an edge is a gradient — blur · Sobel · threshold → %d edge pixels become structure" % edge_px,
                 fontsize=11.5, fontweight="bold", color="#1b2330", y=1.04)
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep17_contours.py
"""
DS17 · Thresholding & Contours → Features. Seeded, offline, reproduces to the digit.
threshold collapses gray → black/white blobs; findContours traces each → a countable, measurable list of objects.
"""
import pathlib
import numpy as np
import cv2
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds17.png"
H, W = 60, 80
RECTS = [(5, 20, 6, 30), (8, 48, 40, 52), (40, 55, 60, 74)]   # (r0,r1,c0,c1) half-open


def build_image():
    rng = np.random.default_rng(7)                # the course contract: same seed → same numbers
    img = np.zeros((H, W), dtype=np.uint8)
    val = int(rng.integers(180, 220))             # foreground brightness (deterministic under seed 7)
    for r0, r1, c0, c1 in RECTS:
        img[r0:r1, c0:c1] = val
    return img, val


def pipeline(image):
    _, mask = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rows = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        rows.append({"x": x, "y": y, "w": w, "h": h, "area": w * h})
    df = pd.DataFrame(rows).sort_values("area", ascending=False).reset_index(drop=True)
    df.insert(0, "obj", ["obj%d" % i for i in range(len(df))])
    return mask, contours, df


if __name__ == "__main__":
    img, val = build_image()
    mask, contours, df = pipeline(img)
    count = len(contours); areas = df["area"].tolist()
    total_fg = int(df["area"].sum()); biggest = int(df["area"].iloc[0])

    print("foreground value      :", val)
    print("contour count         :", count)
    print("feature table (sorted by area, desc):")
    print(df[["obj", "w", "h", "area"]].to_string(index=False))
    print("total foreground area :", total_fg)
    print("largest object area   :", biggest)

    _, _, df2 = pipeline(build_image()[0])
    assert df2["area"].tolist() == areas, "non-deterministic areas!"
    assert biggest == max(areas) == 480 and count == 3 and total_fg == 1050
    print("OK: reproduced; figure == terminal == headline")

    # ---- hero figure: contoured mask + feature table ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#7d8693"})
    fig, (axm, axt) = plt.subplots(1, 2, figsize=(8.6, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
    axm.imshow(mask, cmap="gray", vmin=0, vmax=255)
    for i in range(len(df)):
        x, y, w, h = int(df["x"][i]), int(df["y"][i]), int(df["w"][i]), int(df["h"][i])
        axm.add_patch(Rectangle((x - 0.5, y - 0.5), w, h, fill=False, edgecolor="#FF9F40", lw=2.2))
        axm.text(x, y - 1.5, df["obj"][i], color="#FF9F40", fontsize=9, fontweight="bold")
    axm.set_title("threshold(127) → findContours · %d objects" % count, fontsize=10)
    axm.set_xlabel("x (0–80)"); axm.set_ylabel("y (0–60)")
    # table
    axt.axis("off")
    cell = [[df["obj"][i], str(int(df["w"][i])), str(int(df["h"][i])), str(int(df["area"][i]))] for i in range(len(df))]
    t = axt.table(cellText=cell, colLabels=["obj", "w", "h", "area"], cellLoc="center", loc="center")
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 2.0)
    for j in range(4):
        t[0, j].set_facecolor("#FF4D9D"); t[0, j].set_text_props(color="white", fontweight="bold")
    t[1, 3].set_facecolor("#D6FBE9"); t[1, 3].set_text_props(fontweight="bold")   # largest area
    fig.suptitle("3 objects found — largest is %d px² · total fg %d px²" % (biggest, total_fg),
                 fontsize=12, fontweight="bold", color="#1b2330", y=1.02)
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


In [ ]:
%%writefile ep18_capstone.py
"""
DS18 · Capstone — an ML-ready dataset, end to end. Seeded, offline, reproduces to the digit.
Four libraries, one pipeline: pandas FRAMES → numpy SHAPES → matplotlib SEES → opencv SENSES.
"""
import pathlib
import numpy as np
import pandas as pd

FIG = pathlib.Path(__file__).resolve().parents[1] / "remotion" / "public" / "datafig" / "ds18.png"
N = 200


def build_pipeline():
    rng = np.random.default_rng(7)                       # the course contract: same seed → same numbers
    # 1) FRAME IT (pandas): seeded raw table, inject + clean dirt
    area = rng.normal(1500, 400, N).round(1)
    rooms = rng.integers(1, 6, N)
    price = (area * 210 + rooms * 9000 + rng.normal(0, 25000, N)).round(0)
    df = pd.DataFrame({"area": area, "rooms": rooms, "price": price})
    df.loc[rng.choice(N, 12, replace=False), "area"] = np.nan      # 12 missing
    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)          # 1 duplicate
    raw_rows = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    df["area"] = df["area"].fillna(df["area"].median())
    clean_rows = len(df)
    # 2) SHAPE IT (numpy): featurize + standardize
    X = df[["area", "rooms"]].to_numpy(dtype=np.float64)
    Xz = (X - X.mean(0)) / X.std(0)
    # 3) SEE IT (matplotlib EDA): correlation standardized-area vs price
    corr = float(np.corrcoef(Xz[:, 0], df["price"].to_numpy())[0, 1])
    # 4) SENSE IT (opencv-style, pure numpy): seeded image + augment
    ri = np.random.default_rng(7)
    img = ri.integers(0, 256, (32, 32, 3)).astype(np.uint8)
    aug = np.clip(img[:, ::-1, :].astype(np.int16) + 30, 0, 255).astype(np.uint8)
    return dict(raw_rows=raw_rows, clean_rows=clean_rows, df=df, Xz=Xz,
                feat_mean=Xz.mean(0), feat_std=Xz.std(0), corr=corr,
                img=img, aug=aug, aug_mean=float(aug.mean()), aug_shape=aug.shape)


if __name__ == "__main__":
    r1 = build_pipeline(); r2 = build_pipeline()
    assert np.array_equal(r1["Xz"], r2["Xz"]) and r1["aug_mean"] == r2["aug_mean"]
    print("FRAME  raw rows=%d  -> clean rows=%d  (dropped %d)" % (r1["raw_rows"], r1["clean_rows"], r1["raw_rows"] - r1["clean_rows"]))
    print("SHAPE  feature mean=%s  std=%s" % (np.round(r1["feat_mean"], 3), np.round(r1["feat_std"], 3)))
    print("SEE    corr(area_z, price)=%.3f" % r1["corr"])
    print("SENSE  aug batch shape=%s  mean px=%.2f" % (r1["aug_shape"], r1["aug_mean"]))
    print("REPRODUCE  run #1 == run #2: %s" % np.array_equal(r1["Xz"], r2["Xz"]))
    assert abs(r1["feat_mean"]).max() < 1e-9 and abs(r1["feat_std"] - 1).max() < 1e-9
    assert round(r1["corr"], 3) == 0.924 and round(r1["aug_mean"], 2) == 157.58
    print("OK     pipeline reproduces; std features mean~0 std~1; corr=%.3f" % r1["corr"])

    # ---- hero figure: 2×2 four-stage dashboard ----
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"text.color": "#1b2330", "axes.labelcolor": "#1b2330",
                         "axes.edgecolor": "#7d8693", "xtick.color": "#7d8693", "ytick.color": "#7d8693", "grid.color": "#ececec"})
    df = r1["df"]; Xz = r1["Xz"]
    fig, ax = plt.subplots(2, 2, figsize=(8.6, 6.4))
    # FRAME (pandas, pink)
    ax[0, 0].axis("off"); ax[0, 0].set_title("FRAME IT · pandas", color="#FF4D9D", fontsize=11, fontweight="bold")
    head = df.head(5); cell = [["%.1f" % v for v in head["area"]], [str(int(v)) for v in head["rooms"]], ["%.0f" % v for v in head["price"]]]
    cell = list(map(list, zip(*cell)))
    t = ax[0, 0].table(cellText=cell, colLabels=["area", "rooms", "price"], cellLoc="center", loc="center")
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.4)
    for j in range(3):
        t[0, j].set_facecolor("#FF4D9D"); t[0, j].set_text_props(color="white", fontweight="bold")
    ax[0, 0].text(0.5, -0.02, "201 → 200 rows · dropped 1 dup · 12 NaN filled", transform=ax[0, 0].transAxes, ha="center", fontsize=8, color="#7d8693")
    # SHAPE (numpy, cyan)
    ax[0, 1].hist(Xz[:, 0], bins=20, color="#2BD9FF", edgecolor="#1b2330")
    ax[0, 1].axvline(0, color="#2BB57A", lw=2); ax[0, 1].set_title("SHAPE IT · numpy", color="#1f8fb0", fontsize=11, fontweight="bold")
    ax[0, 1].text(0.5, 0.92, "mean -0.000 · std 1.000", transform=ax[0, 1].transAxes, ha="center", fontsize=9, color="#1b2330")
    # SEE (matplotlib, lime)
    ax[1, 0].scatter(Xz[:, 0], df["price"], s=12, color="#5FBF1F", alpha=0.6)
    sl, ic = np.polyfit(Xz[:, 0], df["price"], 1)
    xs = np.array([Xz[:, 0].min(), Xz[:, 0].max()]); ax[1, 0].plot(xs, sl * xs + ic, color="#3a8f00", lw=2)
    ax[1, 0].set_title("SEE IT · matplotlib", color="#5a9e1f", fontsize=11, fontweight="bold")
    ax[1, 0].text(0.05, 0.88, "corr = %.3f" % r1["corr"], transform=ax[1, 0].transAxes, fontsize=12, fontweight="bold", color="#3a8f00")
    ax[1, 0].set_xlabel("area (z)"); ax[1, 0].set_ylabel("price")
    # SENSE (opencv, orange)
    ax[1, 1].imshow(r1["aug"]); ax[1, 1].set_title("SENSE IT · opencv", color="#C8631F", fontsize=11, fontweight="bold")
    ax[1, 1].set_xticks([]); ax[1, 1].set_yticks([])
    ax[1, 1].set_xlabel("aug (32,32,3) · mean px %.2f" % r1["aug_mean"], fontsize=9, color="#C8631F")
    fig.suptitle("frame → shape → see → sense · one seeded pipeline · corr 0.924 · mean px 157.58",
                 fontsize=11.5, fontweight="bold", color="#1b2330", y=1.0)
    fig.tight_layout()
    FIG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG, dpi=130, facecolor="white", bbox_inches="tight")
    print("saved:", FIG.name)


## DS01 · Arrays, dtypes & Vectorization

*One array op replaces the Python loop — and returns the bit-for-bit identical result.*

In [ ]:
!python ep01_vectorization.py

dtype: float64
shape: (20,)
raw   mean: 46.203  std: 9.1369
array_equal(loop, vec): True
std   mean: 0.0  std: 1.0
reproduce check: PASS
saved: ds01.png


## DS02 · Broadcasting: Shapes That Align Without Copies

*A column (4,1) times a row (3,) becomes a (4,3) table — no loop, no copy.*

In [ ]:
!python ep02_broadcasting.py

a = [5 4 4 5] shape (4,)
b = [3 4 5] shape (3,)
col shape (4, 1)
table shape (4, 3)
[[15 20 25]
 [12 16 20]
 [12 16 20]
 [15 20 25]]
match loop: True
reproduces: True
max cell: 25
max at (row,col): (0, 2) = a[0]*b[2] = 5*5
total sum: 216
saved: ds02.png


## DS03 · Fancy Indexing & Boolean Masks

*A boolean mask is a query; x[mask] is the answer — selection without iteration.*

In [ ]:
!python ep03_masks.py

array:
[[94 62 68 89 57]
 [77 83 22  5 30]
 [28 87 91  0 49]
 [82 13 79 11 46]]
mask:
[[ True  True  True  True  True]
 [ True  True False False False]
 [False  True  True False False]
 [ True False  True False False]]
count > 50 : 11
selected  : [94 62 68 89 57 77 83 87 91 82 79]
sel_sum   : 869
clamped:
[[50 50 50 50 50]
 [50 50 22  5 30]
 [28 50 50  0 49]
 [50 13 50 11 46]]
picked    : [62 49 82] pick_sum : 193
reproducible: True | figure lit == count: True
saved: ds03.png


## DS04 · Linear Algebra: dot, matmul, norms, lstsq

*Fitting a line is one matrix call — least squares is algebra, not a special algorithm.*

In [ ]:
!python ep04_linalg.py

X shape: (40, 2)
rank: 2
slope     = 2.4920
intercept = 0.0983
residual norm = 7.4393
sqrt(lstsq residual) matches norm: True
reproduces: True
figure line: y = 2.492x + 0.0983
saved: ds04.png


## DS05 · DataFrame & Series: The Labeled Array

*A DataFrame is numpy with names and mixed dtypes; head/dtypes/describe read it first.*

In [ ]:
!python ep05_dataframe.py

shape: (6, 5)

head:
   user_id  age  score  active  plan
0     1001   62  59.31   False   pro
1     1002   47  64.54   False   pro
2     1003   50  58.10   False   pro
3     1004   60  70.72   False  team
4     1005   45  86.08   False  team

dtypes:
user_id      int64
age          int64
score      float64
active        bool
plan           str
dtype: object

describe (numeric):
       user_id    age  score
count     6.00   6.00   6.00
mean   1003.50  53.00  67.14
std       1.87   6.93  10.30
min    1001.00  45.00  58.10
25%    1002.25  47.75  60.50
50%    1003.50  52.00  64.32
75%    1004.75  58.50  69.18
max    1006.00  62.00  86.08

reproduce: OK | figure mean(score) == terminal mean(score) == 67.14
saved: ds05.png


## DS06 · Indexing & Selection: loc, iloc, boolean

*Three doors: by label, by position, by condition — pick the right one.*

In [ ]:
!python ep06_indexing.py

   user  plan  age  sessions  spend
0   u00   pro   48        28  46.93
1   u01  team   28        24  40.12
2   u02   pro   59        13   9.22
3   u03  free   36        39  46.59
4   u04  free   38        18  20.63
5   u05  team   39         8  20.66
6   u06  free   42        33  77.22
7   u07  team   41         6  38.16
8   u08   pro   39        34  43.76
9   u09  free   59        24  44.17
10  u10  free   51         4   9.02
11  u11  free   51         1  45.64

.loc[3, 'spend'] (label)  = 46.59

.iloc[0:3, 0:2] (position):
  user  plan
0  u00   pro
1  u01  team
2  u02   pro

boolean (plan=='pro') & (spend>40):
  user plan  age  sessions  spend
0  u00  pro   48        28  46.93
8  u08  pro   39        34  43.76

rows matched = 2
mean spend   = 45.34
.at[3,'spend'] = 46.59   (== .loc? True )

reproduce: OK  | figure == terminal: OK
saved: ds06.png


## DS07 · GroupBy: Split, Apply, Combine

*Split by key, apply an aggregation, combine — raw rows to a ranked feature in one line.*

In [ ]:
!python ep07_groupby.py

rows: 30
cat  value
  C  56.95
  B  36.56
  C  45.42
  C  30.99
  B  37.10
  C  31.58

      mean  count
cat              
B    49.54      8
A    47.13     10
C    41.53     12

top group: B  mean=49.54
rows accounted for: 30
reproduce: OK | figure==terminal: OK
saved: ds07.png


## DS08 · Merge, Join & Reshape: pivot and melt

*Joins align by key (inner vs left); pivot/melt reshape long <-> wide. The totals never lie.*

In [ ]:
!python ep08_merge.py

inner rows : 5
left  rows : 12
dropped by inner (unmatched keys): 7

pivot_table (region x status, count of orders):
status  paid  refund
region              
north      2       0
south      2       1

pivot grand total : 5
melt round-trip total : 5
reproduce check: PASS
saved: ds08.png


## DS09 · Missing Data & Feature Engineering

*The fixed recipe: impute (median) -> encode (get_dummies) -> scale (standardize).*

In [ ]:
!python ep09_features.py

NaNs per column:
age       16
income    24
tenure    10
region     0

Medians used to impute:
age          44.5
income    58888.0
tenure       60.0

Columns after one-hot:
['age', 'income', 'tenure', 'region_east', 'region_north', 'region_south']

income   mean=58206.80  std=13633.69
income_z mean=-0.0000  std=1.0000
income_z min=-3.446  max=2.601

reproduce check: PASS (NaN counts identical on rerun)
saved: ds09.png | holes: 50


## DS10 · Anatomy of a Figure: fig, axes, artists

*A figure is a tree of objects you own — configure the handle, never the global.*

In [ ]:
!python ep10_anatomy.py

figure objects : fig=Figure  ax=Axes
line artist    : Line2D, n_points=12
legend entries : 2
peak y         : 21.357 at x=10.000
lstsq fit      : slope=2.045 intercept=0.678
reproduce check: (12, 21.357, 2.045, 0.678)  (run twice → identical)
saved: ds10.png


## DS11 · EDA Plots: scatter, hist, box

*Plot before you model — EDA shows structure and which feature carries the signal.*

In [ ]:
!python ep11_eda.py

class0  feature1 mean: 1.874  feature2 mean: 1.915
class1  feature1 mean: 3.951  feature2 mean: 4.278
feature2 mean gap (class1 - class0): 2.363
feature1 gap: 2.077   feature2 gap: 2.363
better separator: feature2
reproduce: OK   figure_gap == terminal_gap: 2.363
saved: ds11.png


## DS12 · Visualizing Model Results: confusion matrix, ROC/PR

*Accuracy is one number hiding four — read the matrix, not the average.*

In [ ]:
!python ep12_report.py

class balance: 95 neg, 105 pos
confusion matrix [[TN,FP],[FN,TP]]:
[[84 11]
 [20 85]]
accuracy : 0.845
precision: 0.885
recall   : 0.810
ROC-AUC  : 0.927
reproduces: True
figure TP == terminal TP: 85
saved: ds12.png


## DS13 · Subplots & Styling for Reports

*One figure, four panels, styled once and saved to the digit — report-ready.*

In [ ]:
!python ep13_dashboard.py

PANEL 1  hist   mean      = 48.68
PANEL 2  fit    slope     = 1.94
PANEL 3  bars   top       = A @ 59
PANEL 4  cum    endpoint  = 46
FIGURE   annotated slope  = 1.94
CHECK    figure == terminal: True
CHECK    image bytes identical: True
CHECK    stats identical      : True
saved: ds13.png


## DS14 · Images Are NumPy Arrays: shape, dtype, color spaces

*An image is an (H,W,3) uint8 array — OpenCV is NumPy with a camera-shaped API.*

In [ ]:
!python ep14_image_arrays.py

shape: (64, 64, 3)
dtype: uint8
min/max: 0 255
B mean: 123.36
G mean: 160.08
R mean: 94.24
gray shape: (64, 64)
hsv shape: (64, 64, 3)
gray mean: 136.21
plain avg: 125.89 (gray leans toward G: True )
OK reproduce + figure==terminal
saved: ds14.png


## DS15 · Preprocessing for ML: resize, crop, normalize, dtype

*A model's input is a promise: fixed shape, float32, [0,1]. Preprocessing keeps it.*

In [ ]:
!python ep15_preprocess.py

stage       shape           dtype         min     max     mean
original    (200, 300, 3)   uint8       0.000 255.000  128.426
resized     (128, 128, 3)   uint8       0.000 255.000  128.425
cropped     (96, 96, 3)     uint8       0.000 255.000  129.057
normalized  (96, 96, 3)     float32     0.000   1.000    0.506
reproduce check: identical on re-run -> OK
figure values: norm_max=1.0  norm_mean=0.506
saved: ds15.png


## DS16 · Filtering & Edges: blur, Sobel, Canny

*An edge is a gradient — blur, differentiate, thin, threshold -> structure.*

In [ ]:
!python ep16_edges.py

image shape : (200, 200)
image dtype : uint8
input values: [60, 255]
blur kernel : (5, 5)
canny thresh: (50, 150)
edge values : [0, 255]
edge pixels : 559
grad@edge   : 487.6
grad@flat   : 14.3
reproduce   : identical (edges == edges2)
figure value: 559 == terminal edge pixels  OK
saved: ds16.png


## DS17 · Thresholding & Contours -> Features

*Threshold splits, contours wrap — pixels become countable, measurable features.*

In [ ]:
!python ep17_contours.py

foreground value      : 217
contour count         : 3
feature table (sorted by area, desc):
 obj  w  h  area
obj0 12 40   480
obj1 24 15   360
obj2 14 15   210
total foreground area : 1050
largest object area   : 480
OK: reproduced; figure == terminal == headline
saved: ds17.png


## DS18 · Capstone: An ML-Ready Dataset, End to End

*Frame it, shape it, see it, sense it — one seeded pipeline that reproduces to the digit.*

In [ ]:
!python ep18_capstone.py

FRAME  raw rows=201  -> clean rows=200  (dropped 1)
SHAPE  feature mean=[-0.  0.]  std=[1. 1.]
SEE    corr(area_z, price)=0.924
SENSE  aug batch shape=(32, 32, 3)  mean px=157.58
REPRODUCE  run #1 == run #2: True
OK     pipeline reproduces; std features mean~0 std~1; corr=0.924
saved: ds18.png
